Step1:
generated lables file. 1 for AD and 0 for Healthy


In [1]:
import pandas as pd

# YOUR SUBJECT NAMES HERE - replace with REAL names from your folders
ad_subjects = [
    "sub1_HbO", "sub1_HbR", "sub2_HbO", "sub2_HbR", "sub3_HbO", "sub3_HbR",
    "sub4_HbO", "sub4_HbR", "sub5_HbO", "sub5_HbR", "sub6_HbO", "sub6_HbR",
    "sub7_HbO", "sub7_HbR", "sub8_HbO", "sub8_HbR", "sub9_HbO", "sub9_HbR",
    "sub10_HbO", "sub10_HbR", "sub11_HbO", "sub11_HbR", "sub12_HbO", "sub12_HbR",
    "sub13_HbO", "sub13_HbR", "sub14_HbO", "sub14_HbR", "sub15_HbO", "sub15_HbR",
    "sub16_HbO", "sub16_HbR"
]

healthy_subjects = [
        "h_sub1_HbO", "h_sub1_HbR", "h_sub2_HbO", "h_sub2_HbR", "h_sub3_HbO", "h_sub3_HbR",
    "h_sub4_HbO", "h_sub4_HbR", "h_sub5_HbO", "h_sub5_HbR", "h_sub6_HbO", "h_sub6_HbR",
    "h_sub7_HbO", "h_sub7_HbR", "h_sub8_HbO", "h_sub8_HbR", "h_sub9_HbO", "h_sub9_HbR",
    "h_sub10_HbO", "h_sub10_HbR", "h_sub11_HbO", "h_sub11_HbR", "h_sub12_HbO", "h_sub12_HbR",
    "h_sub13_HbO", "h_sub13_HbR", "h_sub15_HbO", "h_sub15_HbR",
    "h_sub16_HbO", "h_sub16_HbR", "h_sub17_HbO", "h_sub17_HbR", "h_sub18_HbO", "h_sub18_HbR",
    "h_sub19_HbO", "h_sub19_HbR", "h_sub20_HbO", "h_sub20_HbR", "h_sub21_HbO", "h_sub21_HbR",
    "h_sub22_HbO", "h_sub22_HbR", "h_sub23_HbO", "h_sub23_HbR", "h_sub24_HbO", "h_sub24_HbR",
    "h_sub25_HbO", "h_sub25_HbR", "h_sub26_HbO", "h_sub26_HbR", "h_sub27_HbO", "h_sub27_HbR"
]

# Create labels DataFrame
all_subjects = ad_subjects + healthy_subjects
labels = [1] * len(ad_subjects) + [0] * len(healthy_subjects)

labels_df = pd.DataFrame({
    "subject_id": all_subjects,
    "label": labels
})

# Save to Colab's /content/ folder
labels_df.to_csv("/content/subjects_labels.csv", index=False)
print("✅ Labels file created!")
print(labels_df.head(10))
print(f"Total: {len(labels_df)} subjects (AD: {sum(labels)}, Healthy: {len(labels_df)-sum(labels)})")

✅ Labels file created!
  subject_id  label
0   sub1_HbO      1
1   sub1_HbR      1
2   sub2_HbO      1
3   sub2_HbR      1
4   sub3_HbO      1
5   sub3_HbR      1
6   sub4_HbO      1
7   sub4_HbR      1
8   sub5_HbO      1
9   sub5_HbR      1
Total: 84 subjects (AD: 32, Healthy: 52)


In [2]:
import numpy as np
import pandas as pd
import os
from scipy.interpolate import interp1d
from scipy.stats import skew, kurtosis
from scipy.signal import welch

# Your fixed loader (from previous)
def load_subject_tensor(subject_id):
    hbo_path = f"/content/{subject_id}_HbO.csv"
    hbr_path = f"/content/{subject_id}_HbR.csv"


    if not (os.path.exists(hbo_path) and os.path.exists(hbr_path)):
        return None

    # Load with pandas and ensure all values are numeric
    hbo_df = pd.read_csv(hbo_path, header=None).apply(pd.to_numeric, errors='coerce')
    hbr_df = pd.read_csv(hbr_path, header=None).apply(pd.to_numeric, errors='coerce')


    # Convert to NumPy array and transpose
    hbo = hbo_df.values.T
    hbr = hbr_df.values.T


    return np.stack([hbo, hbr], axis=2)

def clean_normalize(X):
    n_ch, n_t, n_c = X.shape
    X_clean = X.copy()

    # NaN interpolation
    for ch in range(n_ch):
        for c in range(n_c):
            ts = X_clean[ch, :, c]
            nan_mask = np.isnan(ts)
            if np.sum(nan_mask) > n_t*0.5:
                return None
            if np.any(nan_mask):
                valid = np.where(~nan_mask)[0]
                if len(valid) >= 2:
                    f = interp1d(valid, ts[valid], kind='linear',
                               bounds_error=False, fill_value="extrapolate")
                    X_clean[ch, :, c][nan_mask] = f(np.where(nan_mask)[0])

    # Z-score
    X_norm = np.zeros_like(X_clean)
    for ch in range(n_ch):
        for c in range(n_c):
            ts = X_clean[ch, :, c]
            X_norm[ch, :, c] = (ts - ts.mean()) / (ts.std() + 1e-8)
    return X_norm

def extract_features(X_norm):
    feats = []
    n_ch, n_t, n_c = X_norm.shape

    # Time + frequency features
    for ch in range(4):
        for c in range(2):
            ts = X_norm[ch, :, c]
            feats.extend([ts.mean(), ts.std(), ts.min(), ts.max(),
                         np.sum(ts**2), skew(ts), kurtosis(ts)])

            f, psd = welch(ts, fs=10.0)
            total = np.trapz(psd, f) + 1e-12
            for lo,hi in [(0.01,0.08),(0.08,0.15),(0.15,0.3)]:
                mask = (f>=lo)&(f<=hi)
                bp = np.trapz(psd[mask], f[mask])
                feats.extend([bp, bp/total])

    # Channel connectivity
    for c in range(2):
        corr = np.corrcoef(X_norm[:,:,c])
        feats.extend(corr[np.triu_indices(4, k=1)])

    return np.array(feats)

# 🔥 PROCESS ALL SUBJECTS
print("🚀 PROCESSING SUBJECTS...")
labels_df = pd.read_csv("/content/subjects_labels.csv")

# Custom function to extract base subject ID correctly
def get_base_subject_id(subject_id_full):
    parts = subject_id_full.split('_')
    if parts[0] == 'h': # Covers healthy subjects like h_sub1_HbO
        return '_'.join(parts[0:2])
    else: # Covers AD subjects like sub1_HbO
        return parts[0]

labels_df['base_subject_id'] = labels_df['subject_id'].apply(get_base_subject_id)
unique_subjects_labels = labels_df.drop_duplicates(subset=['base_subject_id'], keep='first').copy()

X_all, y_all = [], []
# Use unique_subjects_labels for iteration
for i, row in enumerate(unique_subjects_labels.itertuples()):
    sid = row.base_subject_id # This will be like 'sub1', 'h_sub1'
    label = row.label

    X_raw = load_subject_tensor(sid)
    if X_raw is None:
        print(f"[{i+1}/{len(unique_subjects_labels)}] {sid}: ❌ MISSING FILES")
        continue

    X_clean = clean_normalize(X_raw)
    if X_clean is None:
        print(f"[{i+1}/{len(unique_subjects_labels)}] {sid}: ⚠️  TOO MANY NaNs")
        continue

    feats = extract_features(X_clean)
    X_all.append(feats)
    y_all.append(label)

X_data = np.vstack(X_all)
y_data = np.array(y_all)

print(f"\n🎉 COMPLETE!")
print(f"✅ Subjects processed: {len(X_data)}/{len(unique_subjects_labels)}")
print(f"✅ Dataset shape: {X_data.shape}  (subjects × 112 features)")
print(f"✅ AD: {sum(y_data)}, Healthy: {len(y_data)-sum(y_data)}")

🚀 PROCESSING SUBJECTS...


/tmp/ipykernel_14712/4206431116.py:67: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(psd, f) + 1e-12
/tmp/ipykernel_14712/4206431116.py:70: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  bp = np.trapz(psd[mask], f[mask])
/tmp/ipykernel_14712/4206431116.py:67: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(psd, f) + 1e-12
/tmp/ipykernel_14712/4206431116.py:70: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  bp = np.trapz(psd[mask], f[mask])
/tmp/ipykernel_14712/4206431116.py:67: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `s


🎉 COMPLETE!
✅ Subjects processed: 42/42
✅ Dataset shape: (42, 116)  (subjects × 112 features)
✅ AD: 16, Healthy: 26


/tmp/ipykernel_14712/4206431116.py:67: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(psd, f) + 1e-12
/tmp/ipykernel_14712/4206431116.py:70: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  bp = np.trapz(psd[mask], f[mask])
/tmp/ipykernel_14712/4206431116.py:67: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(psd, f) + 1e-12
/tmp/ipykernel_14712/4206431116.py:70: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  bp = np.trapz(psd[mask], f[mask])


### **Feature Selection Strategies for fNIRS & Alzheimer's Detection**

In clinical datasets where the number of features ($p=116$) is larger than the number of subjects ($n=42$), we use feature selection to improve generalizability. Here are the three main types:

#### **1. Filter Methods (Fast & Statistical)**
These evaluate features individually based on their statistical relationship with the target (AD vs. Healthy).
*   **ANOVA F-test / T-tests**: Ranks features by how significantly their means differ between groups.
*   **Correlation-based Selection**: Identifies and removes redundant features that provide the same information as others.
*   **Suitability**: Good as a first pass, but ignores how features might work together (e.g., connectivity between two brain regions).

#### **2. Wrapper Methods (Model-Dependent)**
These use a specific classifier (like Random Forest) to 'wrap' around the selection process.
*   **Recursive Feature Elimination (RFE)**: (Our previous approach) It builds a model, calculates feature importance, and prunes the least important features iteratively.
*   **Forward/Backward Selection**: Adds or subtracts features one-by-one based on the improvement in cross-validation accuracy.
*   **Suitability**: Highly effective for fNIRS because it captures complex interactions between channels.

#### **3. Embedded Methods (Self-Selecting)**
Selection happens automatically as part of the model training.
*   **LASSO (L1 Regularization)**: A linear model that applies a penalty which forces the coefficients of less useful features to zero.
*   **Random Forest / XGBoost Importance**: These models naturally rank features based on how much they contribute to reducing classification error.
*   **Suitability**: Excellent for preventing overfitting in small samples.

### **Why RFE is Strong for Your Task**
For AD detection, we used **RFE with Random Forest**. This is a robust 'Wrapper' method because it doesn't just look at one channel's mean; it looks at how the *combination* of frequency power and connectivity differentiates a patient from a healthy control.

For the type of data we are having,following:

 Recursive Featue elimination(RFE)
 Least Absolute shrinkage and selection operator(LASSO)

 yields optimal results, let's implement them and compare

In [3]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import numpy as np

# Initialize the base estimator
estimator = RandomForestClassifier(n_estimators=100, random_state=42)

# Initialize RFE: selecting the top 20 features
selector = RFE(estimator, n_features_to_select=20, step=5)
selector = selector.fit(X_data, y_data)

# Transform the data
X_selected = selector.transform(X_data)

# Get the names of selected features
selected_indices = np.where(selector.support_)[0]

print(f"✅ RFE Complete!")
print(f"Original feature count: {X_data.shape[1]}")
print(f"Selected feature count: {X_selected.shape[1]}")

✅ RFE Complete!
Original feature count: 116
Selected feature count: 20


In [4]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
import numpy as np

# Embedded methods like LASSO require feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_data)

# Initialize LassoCV
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y_data)

# Identify selected features
selected_mask = lasso.coef_ != 0
lasso_features_indices = np.where(selected_mask)[0]
X_lasso_selected = X_data[:, selected_mask]

print(f"✅ LASSO Complete!")
print(f"Selected feature count: {len(lasso_features_indices)}")

✅ LASSO Complete!
Selected feature count: 9


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def get_loocv_metrics(X, y, model):
    """Helper function to run LOOCV and return metrics."""
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        model.fit(X_train, y_train)
        y_true.append(y_test[0])
        y_pred.append(model.predict(X_test)[0])
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    }

# Define models
candidate_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (RBF)': SVC(kernel='rbf', probability=True)
}

results = []
print("⌛ Running LOOCV for all model and feature set combinations...")

# Iterate through feature sets and models
for set_label, X_set in [('RFE (20)', X_selected), ('LASSO (9)', X_lasso_selected)]:
    for model_name, model in candidate_models.items():
        metrics = get_loocv_metrics(X_set, y_data, model)
        results.append({
            'Feature Set': set_label,
            'Model': model_name,
            **metrics
        })

# Display Results
summary_df = pd.DataFrame(results)
print("\n⅒ FULL PERFORMANCE MATRIX")
display(summary_df.sort_values(by=['Feature Set', 'Accuracy'], ascending=False))

⌛ Running LOOCV for all model and feature set combinations...

⅒ FULL PERFORMANCE MATRIX


,Feature Set,Model,Accuracy,Precision,Recall,F1-Score
0,RFE (20),Random Forest,0.857143,0.916667,0.6875,0.785714
1,RFE (20),Logistic Regression,0.785714,0.769231,0.6250,0.689655
2,RFE (20),SVM (RBF),0.666667,1.000000,0.1250,0.222222
3,LASSO (9),Random Forest,0.738095,0.692308,0.5625,0.620690
4,LASSO (9),Logistic Regression,0.714286,0.700000,0.4375,0.538462
5,LASSO (9),SVM (RBF),0.619048,0.000000,0.0000,0.000000


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

def get_loocv_metrics_custom(X, y, model, threshold=0.5):
    """Run LOOCV with support for custom probability thresholds."""
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        model.fit(X_train, y_train)

        # Get probability for class 1 (AD) if available, otherwise use direct prediction (threshold won't apply)
        if hasattr(model, 'predict_proba'):
            prob = model.predict_proba(X_test)[0, 1]
            y_pred.append(1 if prob >= threshold else 0)
        else:
            # For models without predict_proba, threshold doesn't apply, use direct prediction
            y_pred.append(model.predict(X_test)[0])
        y_true.append(y_test[0])

    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    }

# Define models
candidate_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (RBF)': SVC(kernel='rbf', probability=True) # Ensure probability=True for SVM to use predict_proba
}

results_threshold_0_4 = []
print("⌛ Running LOOCV for all model and feature set combinations with threshold = 0.4...")

# Iterate through feature sets and models
for set_label, X_set in [('RFE (20)', X_selected), ('LASSO (9)', X_lasso_selected)]:
    for model_name, model in candidate_models.items():
        # Use the custom metrics function with the specified threshold
        metrics = get_loocv_metrics_custom(X_set, y_data, model, threshold=0.4)
        results_threshold_0_4.append({
            'Feature Set': set_label,
            'Model': model_name,
            **metrics
        })

# Display Results
summary_df_0_4 = pd.DataFrame(results_threshold_0_4)
print("\n⅒ FULL PERFORMANCE MATRIX (Threshold = 0.4)")
display(summary_df_0_4.sort_values(by=['Feature Set', 'Accuracy'], ascending=False))

⌛ Running LOOCV for all model and feature set combinations with threshold = 0.4...

⅒ FULL PERFORMANCE MATRIX (Threshold = 0.4)


,Feature Set,Model,Accuracy,Precision,Recall,F1-Score
0,RFE (20),Random Forest,0.833333,0.764706,0.8125,0.787879
1,RFE (20),Logistic Regression,0.738095,0.631579,0.7500,0.685714
2,RFE (20),SVM (RBF),0.619048,0.500000,0.1875,0.272727
3,LASSO (9),Random Forest,0.809524,0.750000,0.7500,0.750000
4,LASSO (9),Logistic Regression,0.785714,0.769231,0.6250,0.689655
5,LASSO (9),SVM (RBF),0.285714,0.000000,0.0000,0.000000


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

def get_loocv_metrics_custom(X, y, model, threshold=0.5):
    """Run LOOCV with support for custom probability thresholds."""
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        model.fit(X_train, y_train)

        # Get probability for class 1 (AD)
        prob = model.predict_proba(X_test)[0, 1]
        y_true.append(y_test[0])
        y_pred.append(1 if prob >= threshold else 0)

    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    }

# Focus on improving the top performer (Random Forest + RFE)
print("⌛ Optimizing Random Forest for higher Recall...")

rf_variants = [
    ('RF (Standard)', RandomForestClassifier(n_estimators=100, random_state=42), 0.5),
    ('RF (Balanced Weights)', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42), 0.5),
    ('RF (Threshold=0.4)', RandomForestClassifier(n_estimators=100, random_state=42), 0.4),
    ('RF (Balanced + Threshold=0.35)', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42), 0.35)
]

optimization_results = []
for name, model, thresh in rf_variants:
    metrics = get_loocv_metrics_custom(X_selected, y_data, model, threshold=thresh)
    optimization_results.append({'Config': name, 'Threshold': thresh, **metrics})

opt_df = pd.DataFrame(optimization_results)
print("\n🚀 RECALL OPTIMIZATION RESULTS (using RFE 20 Features)")
display(opt_df.sort_values('Recall', ascending=False))

⌛ Optimizing Random Forest for higher Recall...

🚀 RECALL OPTIMIZATION RESULTS (using RFE 20 Features)


,Config,Threshold,Accuracy,Precision,Recall,F1-Score
3,RF (Balanced + Threshold=0.35),0.35,0.833333,0.695652,1.0000,0.820513
2,RF (Threshold=0.4),0.40,0.833333,0.764706,0.8125,0.787879
0,RF (Standard),0.50,0.857143,0.916667,0.6875,0.785714
1,RF (Balanced Weights),0.50,0.809524,0.833333,0.6250,0.714286


### **Advanced Optimization Strategies**

To move beyond the current results, we will implement:
1. **SMOTE**: To handle the class imbalance (16 AD vs 26 Healthy) by creating synthetic samples.
2. **XGBoost**: A powerful gradient boosting framework.
3. **GridSearchCV**: To find the optimal model parameters.

In [7]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

# Define a pipeline with SMOTE and XGBoost
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('classifier', xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# Define parameters for Grid Search
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0]
}

# Use LOOCV for the search
grid_search = GridSearchCV(pipeline, param_grid, cv=LeaveOneOut(), scoring='f1', n_jobs=-1)
grid_search.fit(X_selected, y_data)

print(f"✅ Best Parameters: {grid_search.best_params_}")
print(f"✅ Best LOOCV F1-Score: {grid_search.best_score_:.4f}")

# Evaluate the best model's full metrics
best_model = grid_search.best_estimator_
final_metrics = get_loocv_metrics(X_selected, y_data, best_model)
display(pd.DataFrame([final_metrics], index=['XGBoost + SMOTE (Optimized)']))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Best Parameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 50, 'classifier__subsample': 1.0}
✅ Best LOOCV F1-Score: 0.2857


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:18:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

,Accuracy,Precision,Recall,F1-Score
XGBoost + SMOTE (Optimized),0.809524,0.75,0.75,0.75


even after implementation of the above listed advanced methods, our previous simpler random forest(RFE-20) with threshold 0.35 remains the superior clinical strategy.

In [8]:
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_validate, GridSearchCV
from sklearn.utils import resample

# Define our baseline model
base_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

def evaluate_with_threshold(X, y, cv_strategy, threshold=0.5):
    """Custom evaluator for different CV strategies with a probability threshold."""
    scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

    for train_idx, test_idx in cv_strategy.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        base_rf.fit(X_tr, y_tr)
        probs = base_rf.predict_proba(X_te)[:, 1]
        preds = (probs >= threshold).astype(int)

        scores['accuracy'].append(accuracy_score(y_te, preds))
        scores['precision'].append(precision_score(y_te, preds, zero_division=0))
        scores['recall'].append(recall_score(y_te, preds, zero_division=0))
        scores['f1'].append(f1_score(y_te, preds, zero_division=0))

    return {k: np.mean(v) for k, v in scores.items()}

# 1. Stratified 5-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 2. Repeated Stratified K-Fold
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

comparison_results = []

for thresh in [0.35, 0.40]:
    # Run SKF
    res_skf = evaluate_with_threshold(X_selected, y_data, skf, threshold=thresh)
    comparison_results.append({'Method': f'Stratified 5-Fold (T={thresh})', **res_skf})

    # Run Repeated SKF
    res_rskf = evaluate_with_threshold(X_selected, y_data, rskf, threshold=thresh)
    comparison_results.append({'Method': f'Repeated Stratified (T={thresh})', **res_rskf})

# 3. Bootstrap Sampling (n=100 iterations)
boot_scores = []
for i in range(100):
    X_b, y_b = resample(X_selected, y_data, stratify=y_data)
    # Using OOB-like logic or just simple split for bootstrap proxy
    base_rf.fit(X_b, y_b)
    # Test on the full original set to see generalization
    p = base_rf.predict_proba(X_selected)[:, 1]
    preds = (p >= 0.35).astype(int)
    boot_scores.append(recall_score(y_data, preds))

comparison_results.append({
    'Method': 'Bootstrap Mean (T=0.35)',
    'accuracy': 0, 'precision': 0, 'recall': np.mean(boot_scores), 'f1': 0
})

# Display summary table
comparison_df = pd.DataFrame(comparison_results)
print("📊 COMPARISON OF VALIDATION METHODS")
display(comparison_df)

📊 COMPARISON OF VALIDATION METHODS


,Method,accuracy,precision,recall,f1
0,Stratified 5-Fold (T=0.35),0.786111,0.653333,0.866667,0.732857
1,Repeated Stratified (T=0.35),0.772778,0.690000,0.840000,0.736714
2,Stratified 5-Fold (T=0.4),0.811111,0.770000,0.816667,0.747619
3,Repeated Stratified (T=0.4),0.775556,0.734333,0.760000,0.718540
4,Bootstrap Mean (T=0.35),0.000000,0.000000,0.943125,0.000000
